In [30]:
!pip install openai pandas tqdm

In [31]:
# 1. Kill any existing vLLM processes to free up GPU memory
!pkill -f vllm

# 2. Start the vLLM server in the background using nohup
# This ensures it keeps running even after the cell execution is finished
!nohup vllm serve facebook/opt-125m --port 8000 > vllm.log 2>&1 &

# 3. Wait and verify that the server is fully operational
import time
import requests

print("⏳ Starting server and loading model (this may take up to 2 minutes)...")

for i in range(20): # Retry loop for ~200 seconds
    try:
        # Check model health status via the API
        r = requests.get("http://localhost:8000/v1/models")
        if r.status_code == 200:
            print("✅ Server is UP and ready for benchmarking!")
            break
    except:
        if i % 3 == 0:
            print(f".. Server is still loading (Attempt {i+1}/20)")
        time.sleep(10)
else:
    print("❌ Server failed to start. Check the last 20 lines of vllm.log for errors:")
    !tail -n 20 vllm.log

⏳ Starting server and loading model (this may take up to 2 minutes)...
.. Server is still loading (Attempt 1/20)
.. Server is still loading (Attempt 4/20)
.. Server is still loading (Attempt 7/20)
✅ Server is UP and ready for benchmarking!


In [32]:
import pandas as pd
import asyncio, time, requests
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm

# 1. Configuration (Ensure the NGROK_URL is up to date)
NGROK_URL = "https://treelike-dacia-fibroplastic.ngrok-free.dev"
client = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
MODEL = "facebook/opt-125m"
CONCURRENCY = 50 # Concurrency level for the stress test

# 2. Load Data
# Ensure the CSV file is uploaded to the /content/ directory
df = pd.read_csv('/content/BurstGPT_without_fails_3.csv').head(200)
requests_data = [{"in": int(row['Request tokens']), "out": int(row['Response tokens'])} for _, row in df.iterrows()]

async def send_request(req_info, semaphore):
    async with semaphore:
        start_time = time.perf_counter()
        try:
            response = await client.completions.create(
                model=MODEL,
                prompt="x " * req_info['in'],
                max_tokens=req_info['out'],
                stream=True,
                timeout=30.0
            )
            ttft = None
            total_tokens = 0
            async for chunk in response:
                if ttft is None:
                    ttft = time.perf_counter() - start_time
                total_tokens += 1

            latency = time.perf_counter() - start_time
            return {
                "status": "success",
                "ttft": ttft,
                "tps": total_tokens/latency,
                "itl": (latency-ttft)/(total_tokens-1) if total_tokens > 1 else 0
            }
        except:
            return {"status": "fail"}

async def run_and_report():
    print(f"🔥 Stressing the GPU with {CONCURRENCY} concurrent users...")
    semaphore = asyncio.Semaphore(CONCURRENCY)
    results = await tqdm.gather(*[send_request(req, semaphore) for req in requests_data])

    success = [r for r in results if r["status"] == "success"]
    fails = len(results) - len(success)

    if success:
        # Calculate real-time average metrics
        avg_ttft = (sum(r['ttft'] for r in success) / len(success)) * 1000
        avg_tps = sum(r['tps'] for r in success) / len(success)
        avg_itl = (sum(r['itl'] for r in success) / len(success)) * 1000

        # Prepare Prometheus metrics format
        metrics = f"""
# HELP ttft_ms Real TTFT from Stress Test
ttft_ms {avg_ttft:.2f}
# HELP throughput_tokens_per_sec Real TPS
throughput_tokens_per_sec {avg_tps:.2f}
# HELP itl_ms Real ITL
itl_ms {avg_itl:.2f}
# HELP failure_count Real Fails
failure_count {fails:.2f}
"""
        # Push metrics to local Pushgateway via ngrok
        endpoint = f"{NGROK_URL}/metrics/job/aiperf"
        requests.post(endpoint, data=metrics, headers={"Content-Type": "text/plain", "ngrok-skip-browser-warning": "true"})

        print(f"\n✅ Metrics successfully pushed to Grafana!")
        print(f"📊 Results: TTFT={avg_ttft:.1f}ms | TPS={avg_tps:.1f} | Fails={fails}")

await run_and_report()

🔥 Stressing the GPU with 50 concurrent users...


100%|██████████| 200/200 [00:14<00:00, 13.48it/s]



✅ Metrics successfully pushed to Grafana!
📊 Results: TTFT=1179.4ms | TPS=36.8 | Fails=8
